In [1]:
%pip install mediapipe 
%pip install opencv-python 
%pip install numpy 
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Imports

In [2]:
import cv2
import json
import numpy as np
from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

Configuração

In [3]:
VIDEO_PATH = "Video/video.mp4"
MODEL_PATH = "Models/pose_landmarker_lite.task"

PREVIEW_OUTPUT = "Video/preview.mp4"
JSON_OUTPUT = "Json/pose_landmarks.json"

Inicialização do MediaPipe

In [4]:
BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
RunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=MODEL_PATH
    ),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
)

Processamento do vídeo

In [5]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(
    PREVIEW_OUTPUT,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

all_frames = []

Extração dos landmarks

In [6]:
with PoseLandmarker.create_from_options(options) as landmarker:

    for frame_idx in tqdm(range(frame_count)):

        ret, frame = cap.read()

        if not ret:
            break

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = int(
            frame_idx * 1000 / fps
        )

        result = landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        frame_landmarks = []

        if result.pose_landmarks:

            pose = result.pose_landmarks[0]

            for lm in pose:

                frame_landmarks.append({
                    "x": float(lm.x),
                    "y": float(lm.y),
                    "z": float(lm.z)
                })

                px = int(lm.x * width)
                py = int(lm.y * height)

                cv2.circle(
                    frame,
                    (px, py),
                    3,
                    (0, 255, 0),
                    -1
                )

        all_frames.append(frame_landmarks)

        writer.write(frame)

cap.release()
writer.release()

100%|█████████▉| 246/247 [00:09<00:00, 26.40it/s]


Salvar JSON

In [7]:
with open(JSON_OUTPUT, "w") as f:
    json.dump(all_frames, f)

print("JSON salvo:", JSON_OUTPUT)

JSON salvo: Json/pose_landmarks.json


Verificar um frame

In [8]:
print("Frames:", len(all_frames))

if len(all_frames):
    print(
        "Landmarks frame 0:",
        len(all_frames[0])
    )

Frames: 246
Landmarks frame 0: 33
